<a href="https://colab.research.google.com/github/shivaprabhub2001-web/speech-driven-gesture-generation/blob/main/gesture_generation_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
path = '/content/drive/MyDrive/ BEAT2_data/processed_cache'
files = os.listdir(path)
print(f"Found {len(files)} files")
print(files[:5])

Found 472 files
['2_scott_0_100_100_processed.npz', '2_scott_0_101_101_processed.npz', '2_scott_0_102_102_processed.npz', '2_scott_0_103_103_processed.npz', '2_scott_0_104_104_processed.npz']


In [ ]:
"""
train_gesture_model.py

Trains a Transformer-encoder + DDPM-diffusion-decoder model to generate
upper-body gesture sequences from speech, using the *_processed.npz
files produced by preprocess_beat.py.

Designed to run inside Google Colab. Paste this into a Colab code cell
(or upload as a .py file and run with `!python train_gesture_model.py`).

Before running, make sure you've already run:
    from google.colab import drive
    drive.mount('/content/drive')
and confirmed the path below is correct.
"""

import os
import glob
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# ---------------------------------------------------------------------
# CONFIG
# ---------------------------------------------------------------------
DATA_DIR = "/content/drive/MyDrive/ BEAT2_data/processed_cache"  # note leading space in " BEAT2_data"
CHECKPOINT_DIR = "/content/drive/MyDrive/ BEAT2_data/checkpoints"
BATCH_SIZE = 8
EPOCHS = 30                 # scoped down for the 1-month timeline; raise later if time allows
LEARNING_RATE = 1e-4
D_MODEL = 256                # Transformer hidden size
N_HEADS = 4
N_ENCODER_LAYERS = 4
DIFFUSION_STEPS = 200        # reduced from typical 1000 for faster training/sampling
MAX_SEQ_LEN = 200            # truncate/pad motion sequences to this many frames
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

os.makedirs(CHECKPOINT_DIR, exist_ok=True)


# ---------------------------------------------------------------------
# DATASET
# ---------------------------------------------------------------------
class GestureDataset(Dataset):
    def __init__(self, data_dir):
        self.files = sorted(glob.glob(os.path.join(data_dir, "*_processed.npz")))
        if len(self.files) == 0:
            raise RuntimeError(f"No processed files found in {data_dir}")
        print(f"Dataset: found {len(self.files)} processed sequences")

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        data = np.load(self.files[idx])
        audio = data["audio_features"]       # (T_audio, 14)
        bert = data["bert_features"]         # (T_text, 768)
        motion = data["motion"]              # (T_motion, joints*3) or similar

        audio = self._fix_len(audio, MAX_SEQ_LEN)
        bert = self._fix_len(bert, MAX_SEQ_LEN)
        motion = self._fix_len(motion, MAX_SEQ_LEN)

        return {
            "audio": torch.tensor(audio, dtype=torch.float32),
            "bert": torch.tensor(bert, dtype=torch.float32),
            "motion": torch.tensor(motion, dtype=torch.float32),
        }

    @staticmethod
    def _fix_len(arr, target_len):
        """Pad with zeros or truncate so every sample has the same length."""
        cur_len = arr.shape[0]
        if cur_len >= target_len:
            return arr[:target_len]
        pad_width = [(0, target_len - cur_len)] + [(0, 0)] * (arr.ndim - 1)
        return np.pad(arr, pad_width, mode="constant")


# ---------------------------------------------------------------------
# MODEL: Transformer encoder -> conditioning -> DDPM decoder
# ---------------------------------------------------------------------
class SpeechEncoder(nn.Module):
    """Fuses audio (MFCC+pitch) and BERT features into one conditioning sequence."""

    def __init__(self, audio_dim=14, bert_dim=768, d_model=D_MODEL, n_heads=N_HEADS, n_layers=N_ENCODER_LAYERS):
        super().__init__()
        self.audio_proj = nn.Linear(audio_dim, d_model)
        self.bert_proj = nn.Linear(bert_dim, d_model)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=d_model * 4, batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)

    def forward(self, audio, bert):
        # both (B, T, dim) -> project to same d_model, average-fuse, encode
        a = self.audio_proj(audio)                    # (B, T_a, d_model)
        b = self.bert_proj(bert)                       # (B, T_b, d_model)
        # align lengths by truncating to the shorter of the two
        min_len = min(a.shape[1], b.shape[1])
        fused = a[:, :min_len] + b[:, :min_len]
        return self.transformer(fused)                 # (B, min_len, d_model)


class DiffusionDecoder(nn.Module):
    """DDPM-style denoiser: predicts noise for every frame, conditioned on
    the per-frame (not pooled) encoder output via cross-attention."""

    def __init__(self, motion_dim, d_model=D_MODEL, n_heads=N_HEADS):
        super().__init__()
        self.motion_dim = motion_dim
        self.time_embed = nn.Sequential(
            nn.Linear(1, d_model), nn.SiLU(), nn.Linear(d_model, d_model)
        )
        self.motion_proj = nn.Linear(motion_dim, d_model)
        self.cross_attn = nn.MultiheadAttention(d_model, n_heads, batch_first=True)
        self.norm1 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_model * 2), nn.SiLU(), nn.Linear(d_model * 2, d_model)
        )
        self.norm2 = nn.LayerNorm(d_model)
        self.out_proj = nn.Linear(d_model, motion_dim)

    def forward(self, noisy_motion, t, cond):
        # noisy_motion: (B, T, motion_dim), cond: (B, T_cond, d_model)
        t_embed = self.time_embed(t.float().unsqueeze(-1)).unsqueeze(1)   # (B, 1, d_model)
        m = self.motion_proj(noisy_motion) + t_embed                       # (B, T, d_model), broadcast time to every frame

        attn_out, _ = self.cross_attn(query=m, key=cond, value=cond)       # (B, T, d_model)
        m = self.norm1(m + attn_out)
        m = self.norm2(m + self.ffn(m))

        return self.out_proj(m)                                            # (B, T, motion_dim) — one noise prediction per frame


def get_beta_schedule(num_steps=DIFFUSION_STEPS):
    return torch.linspace(1e-4, 0.02, num_steps)


# ---------------------------------------------------------------------
# TRAINING LOOP
# ---------------------------------------------------------------------
def train():
    dataset = GestureDataset(DATA_DIR)
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)

    sample = dataset[0]
    motion_dim = sample["motion"].shape[-1]
    print(f"Motion feature dim: {motion_dim}")

    encoder = SpeechEncoder().to(DEVICE)
    decoder = DiffusionDecoder(motion_dim=motion_dim).to(DEVICE)
    betas = get_beta_schedule().to(DEVICE)
    alphas = 1.0 - betas
    alphas_cumprod = torch.cumprod(alphas, dim=0)

    optimizer = torch.optim.AdamW(
        list(encoder.parameters()) + list(decoder.parameters()), lr=LEARNING_RATE
    )
    mse = nn.MSELoss()

    print(f"Training on device: {DEVICE}")
    for epoch in range(1, EPOCHS + 1):
        encoder.train()
        decoder.train()
        total_loss = 0.0

        for batch in loader:
            audio = batch["audio"].to(DEVICE)
            bert = batch["bert"].to(DEVICE)
            motion = batch["motion"].to(DEVICE)

            cond = encoder(audio, bert)

            t = torch.randint(0, DIFFUSION_STEPS, (motion.shape[0],), device=DEVICE)
            noise = torch.randn_like(motion)
            sqrt_alpha = alphas_cumprod[t].sqrt().view(-1, 1, 1)
            sqrt_one_minus_alpha = (1 - alphas_cumprod[t]).sqrt().view(-1, 1, 1)
            noisy_motion = sqrt_alpha * motion + sqrt_one_minus_alpha * noise

            predicted_noise = decoder(noisy_motion, t, cond)
            loss = mse(predicted_noise, noise)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        avg_loss = total_loss / len(loader)
        print(f"Epoch {epoch}/{EPOCHS} — loss: {avg_loss:.4f}")

        if epoch % 5 == 0 or epoch == EPOCHS:
            ckpt_path = os.path.join(CHECKPOINT_DIR, f"model_epoch{epoch}.pt")
            torch.save({
                "encoder": encoder.state_dict(),
                "decoder": decoder.state_dict(),
                "epoch": epoch,
            }, ckpt_path)
            print(f"  saved checkpoint: {ckpt_path}")

    print("Training complete.")


if __name__ == "__main__":
    train()


Dataset: found 472 processed sequences
Motion feature dim: 165
Training on device: cuda
Epoch 1/30 — loss: 1.0335
Epoch 2/30 — loss: 0.9993
Epoch 3/30 — loss: 0.9983
Epoch 4/30 — loss: 0.9963
Epoch 5/30 — loss: 0.9942
  saved checkpoint: /content/drive/MyDrive/ BEAT2_data/checkpoints/model_epoch5.pt
Epoch 6/30 — loss: 0.9866
Epoch 7/30 — loss: 0.9084
Epoch 8/30 — loss: 0.7343
Epoch 9/30 — loss: 0.5695
Epoch 10/30 — loss: 0.4664
  saved checkpoint: /content/drive/MyDrive/ BEAT2_data/checkpoints/model_epoch10.pt
Epoch 11/30 — loss: 0.4089
Epoch 12/30 — loss: 0.3611
Epoch 13/30 — loss: 0.3230
Epoch 14/30 — loss: 0.2957
Epoch 15/30 — loss: 0.2795
  saved checkpoint: /content/drive/MyDrive/ BEAT2_data/checkpoints/model_epoch15.pt
Epoch 16/30 — loss: 0.2504
Epoch 17/30 — loss: 0.2333
Epoch 18/30 — loss: 0.2240
Epoch 19/30 — loss: 0.2099
Epoch 20/30 — loss: 0.2076
  saved checkpoint: /content/drive/MyDrive/ BEAT2_data/checkpoints/model_epoch20.pt
Epoch 21/30 — loss: 0.1719
Epoch 22/30 — loss:

In [ ]:
"""
sample_gesture.py

Loads a trained checkpoint (from train_gesture_model.py) and generates a
gesture motion sequence from a chosen processed sample, using the reverse
diffusion (denoising) process. Saves the generated motion as a .npy file
you can later convert to .bvh for viewing in Blender.

Run this in the SAME Colab notebook, in a new cell, after training.
"""

import os
import glob
import numpy as np
import torch

# Re-use the same model classes and config from training.
# If this is a fresh Colab session, first re-run the training script's
# class definitions (SpeechEncoder, DiffusionDecoder, get_beta_schedule)
# and the CONFIG block before running this cell.

CHECKPOINT_PATH = "/content/drive/MyDrive/ BEAT2_data/checkpoints/model_epoch30.pt"
DATA_DIR = "/content/drive/MyDrive/ BEAT2_data/processed_cache"
OUTPUT_DIR = "/content/drive/MyDrive/ BEAT2_data/generated_samples"
MAX_SEQ_LEN = 200
DIFFUSION_STEPS = 200
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

os.makedirs(OUTPUT_DIR, exist_ok=True)


def fix_len(arr, target_len):
    cur_len = arr.shape[0]
    if cur_len >= target_len:
        return arr[:target_len]
    pad_width = [(0, target_len - cur_len)] + [(0, 0)] * (arr.ndim - 1)
    return np.pad(arr, pad_width, mode="constant")


def load_sample(data_dir, index=0):
    """Loads one held-out sample to condition generation on."""
    files = sorted(glob.glob(os.path.join(data_dir, "*_processed.npz")))
    data = np.load(files[index])
    audio = fix_len(data["audio_features"], MAX_SEQ_LEN)
    bert = fix_len(data["bert_features"], MAX_SEQ_LEN)
    motion_dim = data["motion"].shape[-1]
    text = str(data["text"])
    name = os.path.basename(files[index])
    return audio, bert, motion_dim, text, name


@torch.no_grad()
def generate(encoder, decoder, audio, bert, motion_dim, betas, alphas, alphas_cumprod, seq_len=MAX_SEQ_LEN):
    encoder.eval()
    decoder.eval()

    audio_t = torch.tensor(audio, dtype=torch.float32, device=DEVICE).unsqueeze(0)
    bert_t = torch.tensor(bert, dtype=torch.float32, device=DEVICE).unsqueeze(0)
    cond = encoder(audio_t, bert_t)

    # start from pure noise and iteratively denoise (reverse DDPM process)
    x = torch.randn((1, seq_len, motion_dim), device=DEVICE)

    for step in reversed(range(DIFFUSION_STEPS)):
        t = torch.full((1,), step, device=DEVICE, dtype=torch.long)
        predicted_noise = decoder(x, t, cond)

        alpha_t = alphas[step]
        alpha_cumprod_t = alphas_cumprod[step]
        beta_t = betas[step]

        if step > 0:
            noise = torch.randn_like(x)
        else:
            noise = torch.zeros_like(x)

        x = (1 / alpha_t.sqrt()) * (
            x - ((1 - alpha_t) / (1 - alpha_cumprod_t).sqrt()) * predicted_noise
        ) + beta_t.sqrt() * noise

    return x.squeeze(0).cpu().numpy()  # (seq_len, motion_dim)


def main():
    # --- rebuild model architecture (must match training config) ---
    audio, bert, motion_dim, text, name = load_sample(DATA_DIR, index=0)
    print(f"Conditioning on sample: {name}")
    print(f"Text: {text[:100]}...")

    encoder = SpeechEncoder().to(DEVICE)
    decoder = DiffusionDecoder(motion_dim=motion_dim).to(DEVICE)

    checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
    encoder.load_state_dict(checkpoint["encoder"])
    decoder.load_state_dict(checkpoint["decoder"])
    print(f"Loaded checkpoint from epoch {checkpoint['epoch']}")

    betas = get_beta_schedule().to(DEVICE)
    alphas = 1.0 - betas
    alphas_cumprod = torch.cumprod(alphas, dim=0)

    generated_motion = generate(encoder, decoder, audio, bert, motion_dim, betas, alphas, alphas_cumprod)

    out_path = os.path.join(OUTPUT_DIR, f"generated_{name.replace('.npz', '')}.npy")
    np.save(out_path, generated_motion)
    print(f"Saved generated motion to: {out_path}")
    print(f"Shape: {generated_motion.shape}")


if __name__ == "__main__":
    main()

Conditioning on sample: 2_scott_0_100_100_processed.npz
Text: well i'm from a small town in florida not used to big cities in city life one time i felt culture sh...
Loaded checkpoint from epoch 30
Saved generated motion to: /content/drive/MyDrive/ BEAT2_data/generated_samples/generated_2_scott_0_100_100_processed.npy
Shape: (200, 165)


In [ ]:
"""
evaluate_gesture.py

Runs the trained model on a batch of held-out samples and computes two
evaluation metrics:

  - FGD  (Frechet Gesture Distance): compares the distribution of
    generated motion vs real motion in a learned feature space. Lower
    is better (0 = identical distributions).

  - BeatAlign: measures how well generated motion "beats" (peaks in
    velocity/energy) line up in time with audio beats. Higher is
    better (1 = perfect alignment).

Run this in the SAME Colab notebook, in a new cell, after training AND
after the sample_gesture.py cell (it reuses SpeechEncoder, DiffusionDecoder,
get_beta_schedule, and the generate() function already defined there).
"""

import os
import glob
import numpy as np
import torch
import librosa
from scipy import linalg

# ---------------------------------------------------------------------
CHECKPOINT_PATH = "/content/drive/MyDrive/ BEAT2_data/checkpoints/model_epoch30.pt"
DATA_DIR = "/content/drive/MyDrive/ BEAT2_data/processed_cache"
AUDIO_DIR = None  # not needed here; BeatAlign uses the audio_features already cached
N_EVAL_SAMPLES = 20      # how many held-out sequences to evaluate on
MAX_SEQ_LEN = 200
DIFFUSION_STEPS = 200
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def fix_len(arr, target_len):
    cur_len = arr.shape[0]
    if cur_len >= target_len:
        return arr[:target_len]
    pad_width = [(0, target_len - cur_len)] + [(0, 0)] * (arr.ndim - 1)
    return np.pad(arr, pad_width, mode="constant")


# ---------------------------------------------------------------------
# FGD — Frechet distance between real and generated motion feature stats
# ---------------------------------------------------------------------
def extract_motion_stats_feature(motion):
    """
    Simple statistical feature vector per sequence: mean and std of each
    joint dimension, plus mean frame-to-frame velocity. This acts as the
    'feature space' for the Frechet distance (a full pretrained motion
    encoder is the more standard choice if you have time later).
    """
    velocity = np.diff(motion, axis=0)
    feat = np.concatenate([
        motion.mean(axis=0), motion.std(axis=0),
        velocity.mean(axis=0), velocity.std(axis=0),
    ])
    return feat  # (4 * motion_dim,)


def compute_fgd(real_feats, gen_feats):
    """Standard Frechet Distance between two Gaussians fit to the feature sets."""
    mu_r, sigma_r = real_feats.mean(axis=0), np.cov(real_feats, rowvar=False)
    mu_g, sigma_g = gen_feats.mean(axis=0), np.cov(gen_feats, rowvar=False)

    diff = mu_r - mu_g
    covmean, _ = linalg.sqrtm(sigma_r @ sigma_g, disp=False)
    if np.iscomplexobj(covmean):
        covmean = covmean.real

    fgd = diff @ diff + np.trace(sigma_r + sigma_g - 2 * covmean)
    return float(fgd)


# ---------------------------------------------------------------------
# BeatAlign — how well motion energy peaks line up with audio beats
# ---------------------------------------------------------------------
def motion_beats(motion, fps=30):
    """Frames where motion velocity magnitude is a local peak = 'motion beats'."""
    velocity = np.linalg.norm(np.diff(motion, axis=0), axis=1)
    peaks = []
    for i in range(1, len(velocity) - 1):
        if velocity[i] > velocity[i - 1] and velocity[i] > velocity[i + 1]:
            peaks.append(i)
    return np.array(peaks) / fps  # convert to seconds


def audio_beats_from_features(audio_features, fps=30, sr=16000, hop_length=512):
    """
    We don't have the raw waveform here (only cached MFCC/pitch features),
    so we approximate audio energy from the pitch/MFCC-derived features and
    find peaks the same way as motion. This is a simplification — if you
    have time later, re-run librosa.beat.beat_track on the original .wav
    files for a more standard beat-align calculation.
    """
    energy = np.linalg.norm(audio_features, axis=1)
    peaks = []
    for i in range(1, len(energy) - 1):
        if energy[i] > energy[i - 1] and energy[i] > energy[i + 1]:
            peaks.append(i)
    return np.array(peaks) / fps


def compute_beat_align(motion_beat_times, audio_beat_times, sigma=0.1):
    """
    For each audio beat, find the closest motion beat and score with a
    Gaussian kernel (BeatAlign formulation from Li et al., 2021).
    Score is 0-1, higher = better alignment.
    """
    if len(motion_beat_times) == 0 or len(audio_beat_times) == 0:
        return 0.0
    scores = []
    for ab in audio_beat_times:
        min_dist = np.min(np.abs(motion_beat_times - ab))
        scores.append(np.exp(-(min_dist ** 2) / (2 * sigma ** 2)))
    return float(np.mean(scores))


# ---------------------------------------------------------------------
# MAIN EVALUATION LOOP
# ---------------------------------------------------------------------
@torch.no_grad()
def generate_for_eval(encoder, decoder, audio, bert, motion_dim, betas, alphas, alphas_cumprod, seq_len=MAX_SEQ_LEN):
    encoder.eval()
    decoder.eval()
    audio_t = torch.tensor(audio, dtype=torch.float32, device=DEVICE).unsqueeze(0)
    bert_t = torch.tensor(bert, dtype=torch.float32, device=DEVICE).unsqueeze(0)
    cond = encoder(audio_t, bert_t)

    x = torch.randn((1, seq_len, motion_dim), device=DEVICE)
    for step in reversed(range(DIFFUSION_STEPS)):
        t = torch.full((1,), step, device=DEVICE, dtype=torch.long)
        predicted_noise = decoder(x, t, cond)
        alpha_t = alphas[step]
        alpha_cumprod_t = alphas_cumprod[step]
        beta_t = betas[step]
        noise = torch.randn_like(x) if step > 0 else torch.zeros_like(x)
        x = (1 / alpha_t.sqrt()) * (
            x - ((1 - alpha_t) / (1 - alpha_cumprod_t).sqrt()) * predicted_noise
        ) + beta_t.sqrt() * noise
    return x.squeeze(0).cpu().numpy()


def main():
    files = sorted(glob.glob(os.path.join(DATA_DIR, "*_processed.npz")))
    eval_files = files[-N_EVAL_SAMPLES:]  # use the LAST N as a held-out-ish set
    print(f"Evaluating on {len(eval_files)} samples")

    sample0 = np.load(eval_files[0])
    motion_dim = sample0["motion"].shape[-1]

    encoder = SpeechEncoder().to(DEVICE)
    decoder = DiffusionDecoder(motion_dim=motion_dim).to(DEVICE)
    checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
    encoder.load_state_dict(checkpoint["encoder"])
    decoder.load_state_dict(checkpoint["decoder"])
    print(f"Loaded checkpoint from epoch {checkpoint['epoch']}")

    betas = get_beta_schedule().to(DEVICE)
    alphas = 1.0 - betas
    alphas_cumprod = torch.cumprod(alphas, dim=0)

    real_feats, gen_feats = [], []
    beat_align_scores = []

    for i, f in enumerate(eval_files):
        data = np.load(f)
        audio = fix_len(data["audio_features"], MAX_SEQ_LEN)
        bert = fix_len(data["bert_features"], MAX_SEQ_LEN)
        real_motion = fix_len(data["motion"], MAX_SEQ_LEN)

        gen_motion = generate_for_eval(
            encoder, decoder, audio, bert, motion_dim, betas, alphas, alphas_cumprod
        )

        real_feats.append(extract_motion_stats_feature(real_motion))
        gen_feats.append(extract_motion_stats_feature(gen_motion))

        m_beats = motion_beats(gen_motion)
        a_beats = audio_beats_from_features(audio)
        score = compute_beat_align(m_beats, a_beats)
        beat_align_scores.append(score)

        print(f"  [{i+1}/{len(eval_files)}] {os.path.basename(f)} — BeatAlign: {score:.3f}")

    real_feats = np.stack(real_feats)
    gen_feats = np.stack(gen_feats)

    fgd_score = compute_fgd(real_feats, gen_feats)
    mean_beat_align = float(np.mean(beat_align_scores))

    print("\n===== EVALUATION RESULTS =====")
    print(f"Samples evaluated : {len(eval_files)}")
    print(f"FGD               : {fgd_score:.4f}  (lower is better)")
    print(f"BeatAlign (mean)  : {mean_beat_align:.4f}  (higher is better, max 1.0)")
    print("===============================")
    print("\nNote for your Results chapter: report both numbers together with")
    print("N (sample count) and note this uses a statistical-feature FGD rather")
    print("than a pretrained motion autoencoder — state this as a limitation.")


if __name__ == "__main__":
    main()

Evaluating on 20 samples
Loaded checkpoint from epoch 30
  [1/20] 7_sophie_0_93_93_processed.npz — BeatAlign: 0.916
  [2/20] 7_sophie_0_94_94_processed.npz — BeatAlign: 0.921
  [3/20] 7_sophie_0_95_95_processed.npz — BeatAlign: 0.941
  [4/20] 7_sophie_0_96_96_processed.npz — BeatAlign: 0.936
  [5/20] 7_sophie_0_97_97_processed.npz — BeatAlign: 0.925
  [6/20] 7_sophie_0_98_98_processed.npz — BeatAlign: 0.907
  [7/20] 7_sophie_0_99_99_processed.npz — BeatAlign: 0.932
  [8/20] 7_sophie_0_9_9_processed.npz — BeatAlign: 0.894
  [9/20] 7_sophie_1_10_10_processed.npz — BeatAlign: 0.939
  [10/20] 7_sophie_1_11_11_processed.npz — BeatAlign: 0.919
  [11/20] 7_sophie_1_12_12_processed.npz — BeatAlign: 0.931
  [12/20] 7_sophie_1_1_1_processed.npz — BeatAlign: 0.918
  [13/20] 7_sophie_1_2_2_processed.npz — BeatAlign: 0.899
  [14/20] 7_sophie_1_3_3_processed.npz — BeatAlign: 0.926
  [15/20] 7_sophie_1_4_4_processed.npz — BeatAlign: 0.900
  [16/20] 7_sophie_1_5_5_processed.npz — BeatAlign: 0.922
  [1

/tmp/ipykernel_2177/1185296088.py:69: DeprecationWarning: The `disp` argument is deprecated and will be removed in SciPy 1.18.0.
  covmean, _ = linalg.sqrtm(sigma_r @ sigma_g, disp=False)


In [ ]:
"""
render_clips.py

Renders a handful of generated motion sequences as simple skeleton
animation videos (.mp4) that you can embed in a Google Form for the
perceptual study. No Blender needed — uses matplotlib.

Run this in the SAME Colab notebook, in a new cell, after the training
and sampling cells (it reuses SpeechEncoder, DiffusionDecoder,
get_beta_schedule, and generate()).
"""

import os
import glob
import numpy as np
import torch
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, FFMpegWriter

# ---------------------------------------------------------------------
CHECKPOINT_PATH = "/content/drive/MyDrive/ BEAT2_data/checkpoints/model_epoch30.pt"
DATA_DIR = "/content/drive/MyDrive/ BEAT2_data/processed_cache"
OUTPUT_DIR = "/content/drive/MyDrive/ BEAT2_data/study_clips"
N_CLIPS = 6            # how many clips to render for the perceptual study
CLIP_SECONDS = 10      # trim each clip to this many seconds
FPS = 20
MAX_SEQ_LEN = 200
DIFFUSION_STEPS = 200
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

os.makedirs(OUTPUT_DIR, exist_ok=True)


def fix_len(arr, target_len):
    cur_len = arr.shape[0]
    if cur_len >= target_len:
        return arr[:target_len]
    pad_width = [(0, target_len - cur_len)] + [(0, 0)] * (arr.ndim - 1)
    return np.pad(arr, pad_width, mode="constant")


@torch.no_grad()
def generate_motion(encoder, decoder, audio, bert, motion_dim, betas, alphas, alphas_cumprod, seq_len=MAX_SEQ_LEN):
    encoder.eval()
    decoder.eval()
    audio_t = torch.tensor(audio, dtype=torch.float32, device=DEVICE).unsqueeze(0)
    bert_t = torch.tensor(bert, dtype=torch.float32, device=DEVICE).unsqueeze(0)
    cond = encoder(audio_t, bert_t)
    x = torch.randn((1, seq_len, motion_dim), device=DEVICE)
    for step in reversed(range(DIFFUSION_STEPS)):
        t = torch.full((1,), step, device=DEVICE, dtype=torch.long)
        predicted_noise = decoder(x, t, cond)
        alpha_t = alphas[step]
        alpha_cumprod_t = alphas_cumprod[step]
        beta_t = betas[step]
        noise = torch.randn_like(x) if step > 0 else torch.zeros_like(x)
        x = (1 / alpha_t.sqrt()) * (
            x - ((1 - alpha_t) / (1 - alpha_cumprod_t).sqrt()) * predicted_noise
        ) + beta_t.sqrt() * noise
    return x.squeeze(0).cpu().numpy()


def motion_to_2d_points(frame, motion_dim):
    """
    Reshape a flat motion vector into (n_joints, coords) for plotting.
    SMPLX-flame pose params aren't literally XYZ joint positions, so this
    is a simplified 2D projection for visualization purposes only — good
    enough to show relative movement to raters, not for biomechanical
    accuracy. If you later convert to real BVH/joint positions, swap this
    function out.
    """
    # assume the vector can be split into pairs (x, y) — pad if odd
    n = motion_dim // 2
    pts = frame[: n * 2].reshape(n, 2)
    return pts


def render_clip(motion, out_path, fps=FPS):
    n_frames, motion_dim = motion.shape
    pts0 = motion_to_2d_points(motion[0], motion_dim)

    fig, ax = plt.subplots(figsize=(4, 5))
    ax.set_xlim(pts0[:, 0].min() - 1, pts0[:, 0].max() + 1)
    ax.set_ylim(pts0[:, 1].min() - 1, pts0[:, 1].max() + 1)
    ax.set_title("Generated gesture (skeleton preview)")
    ax.axis("off")
    scatter = ax.scatter([], [], s=40, c="royalblue")

    def update(i):
        pts = motion_to_2d_points(motion[i], motion_dim)
        scatter.set_offsets(pts)
        return (scatter,)

    n_show = min(n_frames, CLIP_SECONDS * fps)
    anim = FuncAnimation(fig, update, frames=n_show, interval=1000 / fps, blit=True)

    writer = FFMpegWriter(fps=fps)
    anim.save(out_path, writer=writer)
    plt.close(fig)


def main():
    files = sorted(glob.glob(os.path.join(DATA_DIR, "*_processed.npz")))
    chosen_files = files[-N_CLIPS:]  # same held-out slice used in evaluation

    sample0 = np.load(chosen_files[0])
    motion_dim = sample0["motion"].shape[-1]

    encoder = SpeechEncoder().to(DEVICE)
    decoder = DiffusionDecoder(motion_dim=motion_dim).to(DEVICE)
    checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
    encoder.load_state_dict(checkpoint["encoder"])
    decoder.load_state_dict(checkpoint["decoder"])
    print(f"Loaded checkpoint from epoch {checkpoint['epoch']}")

    betas = get_beta_schedule().to(DEVICE)
    alphas = 1.0 - betas
    alphas_cumprod = torch.cumprod(alphas, dim=0)

    for i, f in enumerate(chosen_files):
        data = np.load(f)
        audio = fix_len(data["audio_features"], MAX_SEQ_LEN)
        bert = fix_len(data["bert_features"], MAX_SEQ_LEN)

        gen_motion = generate_motion(encoder, decoder, audio, bert, motion_dim, betas, alphas, alphas_cumprod)

        name = os.path.basename(f).replace("_processed.npz", "")
        out_path = os.path.join(OUTPUT_DIR, f"clip_{i+1}_{name}.mp4")
        render_clip(gen_motion, out_path)
        print(f"  [{i+1}/{len(chosen_files)}] saved: {out_path}")

    print(f"\nDone. {len(chosen_files)} clips saved to '{OUTPUT_DIR}'.")
    print("Next: open that Drive folder, right-click each clip -> Get link ->")
    print("set to 'Anyone with the link', then paste those links into your")
    print("Google Form (one per question) for the perceptual study.")


if __name__ == "__main__":
    main()

Loaded checkpoint from epoch 30
  [1/6] saved: /content/drive/MyDrive/ BEAT2_data/study_clips/clip_1_7_sophie_1_4_4.mp4
  [2/6] saved: /content/drive/MyDrive/ BEAT2_data/study_clips/clip_2_7_sophie_1_5_5.mp4
  [3/6] saved: /content/drive/MyDrive/ BEAT2_data/study_clips/clip_3_7_sophie_1_6_6.mp4
  [4/6] saved: /content/drive/MyDrive/ BEAT2_data/study_clips/clip_4_7_sophie_1_7_7.mp4
  [5/6] saved: /content/drive/MyDrive/ BEAT2_data/study_clips/clip_5_7_sophie_1_8_8.mp4
  [6/6] saved: /content/drive/MyDrive/ BEAT2_data/study_clips/clip_6_7_sophie_1_9_9.mp4

Done. 6 clips saved to '/content/drive/MyDrive/ BEAT2_data/study_clips'.
Next: open that Drive folder, right-click each clip -> Get link ->
set to 'Anyone with the link', then paste those links into your
Google Form (one per question) for the perceptual study.
